<a href="https://colab.research.google.com/github/Fu-Pei-Yin/Deep-Generative-Mode/blob/week5/%E4%BD%BF%E7%94%A8_Seq2Seq%E7%94%9F%E6%88%90%E3%80%8C%E6%9C%AA%E4%BE%86%E5%AD%B8%E7%BF%92%E8%A1%8C%E7%82%BA%E5%BA%8F%E5%88%97%E3%80%8D%E6%95%B8%E6%93%9A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# 設置隨機種子
torch.manual_seed(42)
np.random.seed(42)

# 檢查GPU可用性
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 上傳資料集
print("請上傳以下檔案：studentInfo.csv, studentVle.csv, studentAssessment.csv")
uploaded = files.upload()

# 讀取資料
student_info = pd.read_csv('studentInfo.csv')
student_vle = pd.read_csv('studentVle.csv')
student_assessment = pd.read_csv('studentAssessment.csv')

print("資料集載入完成!")
print(f"studentInfo 形狀: {student_info.shape}")
print(f"studentVle 形狀: {student_vle.shape}")
print(f"studentAssessment 形狀: {student_assessment.shape}")

# 資料預處理函數
def preprocess_data(student_info, student_vle, student_assessment):
    """預處理資料並建立每週序列"""

    # 處理 studentVle - 計算每週點擊數
    weekly_clicks = student_vle.groupby(['id_student', 'date']).agg({
        'sum_click': 'sum'
    }).reset_index()

    # 處理 studentAssessment - 計算提交次數和分數
    weekly_assessments = student_assessment.groupby(['id_student', 'date_submitted']).agg({
        'id_assessment': 'count',
        'score': 'mean'
    }).reset_index()
    weekly_assessments.columns = ['id_student', 'date', 'submit_cnt', 'avg_score']

    # 合併資料
    merged_data = pd.merge(weekly_clicks, weekly_assessments,
                          on=['id_student', 'date'], how='left')
    merged_data.fillna(0, inplace=True)

    # 計算累積平均分數
    merged_data['avg_score_sofar'] = merged_data.groupby('id_student')['avg_score'].cumsum() / (
        merged_data.groupby('id_student').cumcount() + 1)

    # 計算點擊數差分
    merged_data['clicks_diff1'] = merged_data.groupby('id_student')['sum_click'].diff().fillna(0)

    # 是否有提交作業
    merged_data['has_submit'] = (merged_data['submit_cnt'] > 0).astype(int)

    return merged_data

# 預處理資料
processed_data = preprocess_data(student_info, student_vle, student_assessment)
print("資料預處理完成!")

# 建立序列資料集
def create_sequences(data, past_weeks=4, future_weeks=2):
    """建立過去4週預測未來2週的序列"""
    sequences = []
    targets = []
    student_ids = []

    for student_id in data['id_student'].unique():
        student_data = data[data['id_student'] == student_id].sort_values('date')

        if len(student_data) < past_weeks + future_weeks:
            continue

        # 特徵列
        feature_cols = ['sum_click', 'submit_cnt', 'avg_score_sofar', 'clicks_diff1', 'has_submit']

        for i in range(len(student_data) - past_weeks - future_weeks + 1):
            # 過去序列
            past_seq = student_data.iloc[i:i+past_weeks][feature_cols].values
            # 未來目標 (只預測點擊數)
            future_target = student_data.iloc[i+past_weeks:i+past_weeks+future_weeks]['sum_click'].values

            sequences.append(past_seq)
            targets.append(future_target)
            student_ids.append(student_id)

    return np.array(sequences, dtype=np.float32), np.array(targets, dtype=np.float32), student_ids

# 建立序列
X, y, student_ids = create_sequences(processed_data)
print(f"序列資料形狀: X {X.shape}, y {y.shape}")

# 資料標準化
scaler_X = StandardScaler()
scaler_y = StandardScaler()

# 重塑資料以進行標準化
X_reshaped = X.reshape(-1, X.shape[-1])
y_reshaped = y.reshape(-1, 1)

X_scaled = scaler_X.fit_transform(X_reshaped).reshape(X.shape)
y_scaled = scaler_y.fit_transform(y_reshaped).reshape(y.shape)

print("資料標準化完成!")

# 按學生ID分割資料集
unique_students = list(set(student_ids))
train_students, temp_students = train_test_split(unique_students, test_size=0.3, random_state=42)
valid_students, test_students = train_test_split(temp_students, test_size=0.5, random_state=42)

# 建立分割索引
train_mask = [sid in train_students for sid in student_ids]
valid_mask = [sid in valid_students for sid in student_ids]
test_mask = [sid in test_students for sid in student_ids]

X_train, y_train = X_scaled[train_mask], y_scaled[train_mask]
X_valid, y_valid = X_scaled[valid_mask], y_scaled[valid_mask]
X_test, y_test = X_scaled[test_mask], y_scaled[test_mask]

print(f"訓練集: {X_train.shape}, 驗證集: {X_valid.shape}, 測試集: {X_test.shape}")

# 建立 PyTorch Dataset
class StudentSequenceDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = torch.FloatTensor(sequences)
        self.targets = torch.FloatTensor(targets)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

# 建立 DataLoader
batch_size = 128
train_dataset = StudentSequenceDataset(X_train, y_train)
valid_dataset = StudentSequenceDataset(X_valid, y_valid)
test_dataset = StudentSequenceDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("DataLoader 建立完成!")

# Seq2Seq LSTM 模型
class Seq2SeqLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(Seq2SeqLSTM, self).__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

    def forward(self, x):
        # Encoder
        _, (hidden, cell) = self.encoder(x)

        # Decoder - 使用encoder的最後狀態，預測未來2個時間步
        decoder_input = torch.zeros(x.size(0), 2, self.hidden_dim).to(x.device)
        decoder_output, _ = self.decoder(decoder_input, (hidden, cell))

        # 全連接層
        output = self.fc(decoder_output)
        return output

# Seq2Seq VAE 模型
class Seq2SeqVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, output_dim, num_layers=1):
        super(Seq2SeqVAE, self).__init__()

        # Encoder
        self.encoder_lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.decoder_lstm = nn.LSTM(latent_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_decoder = nn.Linear(hidden_dim, output_dim)

        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim

    def encode(self, x):
        _, (hidden, _) = self.encoder_lstm(x)
        hidden_last = hidden[-1]  # 取最後一層的最後隱藏狀態
        mu = self.fc_mu(hidden_last)
        logvar = self.fc_logvar(hidden_last)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        # 將潛在變量擴展為序列
        decoder_input = z.unsqueeze(1).repeat(1, 2, 1)  # 未來2週
        decoder_output, _ = self.decoder_lstm(decoder_input)
        output = self.fc_decoder(decoder_output)
        return output

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar

# 訓練函數
def train_lstm(model, train_loader, valid_loader, epochs=30, lr=1e-3):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    train_losses = []
    valid_losses = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # 驗證
        model.eval()
        valid_loss = 0
        with torch.no_grad():
            for batch_X, batch_y in valid_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                valid_loss += loss.item()

        train_loss /= len(train_loader)
        valid_loss /= len(valid_loader)
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        if (epoch + 1) % 5 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}')

    return train_losses, valid_losses

def train_vae(model, train_loader, valid_loader, epochs=30, lr=1e-3, beta=0.1):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    valid_losses = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            optimizer.zero_grad()
            recon_x, mu, logvar = model(batch_X)

            # VAE loss: reconstruction + KL divergence
            recon_loss = nn.MSELoss()(recon_x, batch_y)
            kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            kld_loss /= batch_X.size(0)  # 平均KL損失

            loss = recon_loss + beta * kld_loss
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # 驗證
        model.eval()
        valid_loss = 0
        with torch.no_grad():
            for batch_X, batch_y in valid_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                recon_x, mu, logvar = model(batch_X)

                recon_loss = nn.MSELoss()(recon_x, batch_y)
                kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
                kld_loss /= batch_X.size(0)

                loss = recon_loss + beta * kld_loss
                valid_loss += loss.item()

        train_loss /= len(train_loader)
        valid_loss /= len(valid_loader)
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        if (epoch + 1) % 5 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}')

    return train_losses, valid_losses

# 初始化模型
input_dim = 5  # sum_click, submit_cnt, avg_score_sofar, clicks_diff1, has_submit
hidden_dim = 64
latent_dim = 16
output_dim = 2  # 未來2週的點擊數

lstm_model = Seq2SeqLSTM(input_dim, hidden_dim, output_dim)
vae_model = Seq2SeqVAE(input_dim, hidden_dim, latent_dim, output_dim)

print("模型初始化完成!")
print(f"LSTM 模型參數數量: {sum(p.numel() for p in lstm_model.parameters())}")
print(f"VAE 模型參數數量: {sum(p.numel() for p in vae_model.parameters())}")

# 訓練模型
print("開始訓練 LSTM 模型...")
lstm_train_losses, lstm_valid_losses = train_lstm(lstm_model, train_loader, valid_loader, epochs=30)

print("\n開始訓練 VAE 模型...")
vae_train_losses, vae_valid_losses = train_vae(vae_model, train_loader, valid_loader, epochs=30, beta=0.1)

# 評估函數
def evaluate_models(lstm_model, vae_model, test_loader, num_samples=20):
    lstm_model.eval()
    vae_model.eval()

    lstm_predictions = []
    vae_samples = []  # 儲存多個樣本
    ground_truth = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            # LSTM 預測
            lstm_pred = lstm_model(batch_X)
            lstm_predictions.append(lstm_pred.cpu().numpy())

            # VAE 生成多個樣本
            batch_vae_samples = []
            for _ in range(num_samples):
                vae_recon, _, _ = vae_model(batch_X)
                batch_vae_samples.append(vae_recon.cpu().numpy())

            vae_samples.append(np.array(batch_vae_samples))
            ground_truth.append(batch_y.cpu().numpy())

    lstm_predictions = np.concatenate(lstm_predictions)
    vae_samples = np.concatenate(vae_samples, axis=1)  # shape: (num_samples, batch_size, 2)
    ground_truth = np.concatenate(ground_truth)

    return lstm_predictions, vae_samples, ground_truth

# 評估模型
print("評估模型中...")
lstm_pred, vae_samples, y_true = evaluate_models(lstm_model, vae_model, test_loader)

# 反標準化以獲得原始尺度
lstm_pred_original = scaler_y.inverse_transform(lstm_pred.reshape(-1, 1)).reshape(lstm_pred.shape)
vae_samples_original = scaler_y.inverse_transform(vae_samples.reshape(-1, 1)).reshape(vae_samples.shape)
y_true_original = scaler_y.inverse_transform(y_true.reshape(-1, 1)).reshape(y_true.shape)

# 計算評估指標
def calculate_metrics(lstm_pred, vae_samples, y_true):
    # LSTM MSE
    lstm_mse = np.mean((lstm_pred - y_true) ** 2)

    # VAE Best-of-N MSE
    best_of_n_mse = []
    for i in range(len(y_true)):
        sample_mses = np.mean((vae_samples[:, i, :] - y_true[i]) ** 2, axis=1)
        best_mse = np.min(sample_mses)
        best_of_n_mse.append(best_mse)
    best_of_n_mse = np.mean(best_of_n_mse)

    # Diversity (標準差)
    diversity = np.std(vae_samples, axis=0)  # 每個樣本點的標準差
    avg_diversity = np.mean(diversity)

    # Coverage (比例)
    coverage_scores = []
    for i in range(len(y_true)):
        true_value = y_true[i]
        samples = vae_samples[:, i, :]

        # 檢查真實值是否在樣本範圍內
        min_samples = np.min(samples, axis=0)
        max_samples = np.max(samples, axis=0)
        covered = np.logical_and(true_value >= min_samples, true_value <= max_samples)
        coverage_rate = np.mean(covered)
        coverage_scores.append(coverage_rate)

    coverage = np.mean(coverage_scores)

    return {
        'LSTM_MSE': lstm_mse,
        'VAE_Best_of_N_MSE': best_of_n_mse,
        'VAE_Diversity': avg_diversity,
        'VAE_Coverage': coverage
    }

metrics = calculate_metrics(lstm_pred_original, vae_samples_original, y_true_original)
print("\n=== 評估結果 ===")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

# 可視化結果
def plot_comparison(lstm_pred, vae_samples, y_true, num_examples=5):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for i in range(min(num_examples, 6)):
        ax = axes[i]

        # 真實值
        true_weeks = [1, 2]
        ax.plot(true_weeks, y_true[i], 'go-', linewidth=2, label='Ground Truth', markersize=8)

        # LSTM 預測
        ax.plot(true_weeks, lstm_pred[i], 'ro-', linewidth=2, label='LSTM Prediction', markersize=8)

        # VAE 樣本 (半透明)
        for j in range(vae_samples.shape[0]):
            ax.plot(true_weeks, vae_samples[j, i], 'b-', alpha=0.3, linewidth=0.5)

        # VAE 平均
        vae_mean = np.mean(vae_samples[:, i, :], axis=0)
        ax.plot(true_weeks, vae_mean, 'c--', linewidth=2, label='VAE Mean')

        ax.set_xlabel('Future Weeks')
        ax.set_ylabel('Clicks')
        ax.set_title(f'Sample {i+1}')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# 繪製訓練損失
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(lstm_train_losses, label='LSTM Train Loss')
plt.plot(lstm_valid_losses, label='LSTM Valid Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LSTM Training History')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(vae_train_losses, label='VAE Train Loss')
plt.plot(vae_valid_losses, label='VAE Valid Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE Training History')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 繪製預測比較
print("繪製預測比較圖...")
plot_comparison(lstm_pred_original[:6], vae_samples_original[:, :6, :], y_true_original[:6])

# 輸出分析結果
print("\n=== 模型分析 ===")
print("1. 單一路徑準確度:")
print(f"   - LSTM MSE: {metrics['LSTM_MSE']:.4f}")
print(f"   - VAE Best-of-N MSE: {metrics['VAE_Best_of_N_MSE']:.4f}")

# 保存模型
torch.save(lstm_model.state_dict(), 'lstm_model.pth')
torch.save(vae_model.state_dict(), 'vae_model.pth')
print("\n模型已保存!")